# FIFA World Cup Tournament Scheduling Optimisation

This notebook implements a three-stage optimisation framework for designing the FIFA World Cup tournament structure. The modelling framework follows the prescriptive analytics approach proposed by Ghoniem et al. (2017) for tournament planning and stadium allocation.

The objective is to design a tournament that is:

- **Competitively balanced**, ensuring that groups have similar strength based on FIFA ranking points
- **Commercially attractive**, by distributing high-demand matches across the tournament schedule
- **Operationally efficient**, by allocating matches to stadiums according to stadium capacity and expected spectator demand

The optimisation process is divided into three models.

---

### Model 1 – Balanced Group Formation

Teams are allocated to groups in order to **maximize the minimum group strength**, where group strength is measured by the total FIFA ranking points of teams in the group. This ensures that the weakest group is as strong as possible, leading to competitively balanced groups while satisfying FIFA draw constraints such as pot structure and confederation restrictions.

---

### Model 2 – Group Letter Assignment

The balanced groups obtained from Model 1 are assigned to tournament group letters (A–L). The objective is to distribute match popularity across the tournament schedule by balancing the popularity of **row-sets**, which represent sets of matches played at the same stage of the tournament.

---

### Model 3 – Stadium Assignment

Finally, matches are assigned to stadiums to **maximize the total popularity value** of the tournament schedule. The popularity value of each assignment is defined as the product of the stadium capacity and the popularity of the corresponding row-set.

---

Each model uses the outputs of the previous stage as inputs, forming an integrated optimisation framework for tournament scheduling and stadium allocation.

# Data Sources

To implement the optimisation framework, several datasets are required. These datasets provide information about teams, tournament structure, spectator demand, and stadium capacity. The data used in this study was collected from publicly available sources related to the FIFA World Cup.

The primary data sources include:

**FIFA World Cup tournament information**

https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026

This source provides information about the tournament structure, participating nations, and official tournament format.

**Official tournament draw procedures**

https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/procedures-pots-final-draw

This source describes the official draw rules, including pot structure and confederation restrictions used when allocating teams to groups.

**FIFA World Rankings**

https://inside.fifa.com/fifa-world-ranking/men

FIFA ranking points are used as a proxy for team strength when forming balanced groups.

**Official 2026 World Cup stadium information**

https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/articles/stadium-information-details

This source provides seating capacity information for the stadiums hosting the tournament.

**Confederation and qualification information**

https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/qualifiers

This source identifies the confederation membership of each qualifying nation.

**Historical attendance data**

https://footystats.org/international/fifa-world-cup-2018-russia/attendance

Attendance data from the 2018 FIFA World Cup is used to estimate spectator demand and compute spectator indices.

---

# Data Inputs

The optimisation models require several categories of data.

## Team Data

For each participating nation, the following attributes are collected:

- FIFA ranking points  
- Confederation membership  
- Pot assignment based on ranking  
- Spectator index (SI)

The spectator index represents the relative popularity of a national team and approximates the expected proportion of stadium seats that would be filled by supporters of that team.

---

## Match Schedule

The tournament schedule defines the structure of the group stage matches. This includes:

- Row-set structure (sets of matches played simultaneously)
- Match slots within the schedule
- Rank pairings between teams within each group

These elements determine which teams play against each other during the group stage and are required to evaluate match popularity across the tournament schedule.

---

## Stadium Data

For each stadium hosting the tournament, the following information is required:

- Stadium name
- Seating capacity

Stadium capacity is used to estimate the potential number of spectators that can attend a match and is therefore used in the stadium assignment model.

# Model 1 – Balanced Group Formation


The objective of Model 1 is to allocate teams to groups such that the **minimum group strength (measured by FIFA ranking points)** is maximized, while satisfying FIFA tournament draw constraints. By maximizing the minimum group strength, the model ensures that the weakest group is as strong as possible, resulting in competitively balanced groups.

---

### Tournament Structure and Data Preparation

For the upcoming FIFA World Cup, the tournament consists of **48 teams forming 12 groups**, with each group containing four teams.

Teams are first sorted in **descending order of FIFA ranking points**. Based on this ranking, teams are divided into **four pots**, each containing **12 teams**. The pot assignment is determined directly from the ranking order.

---

### Sets

- $T$ : set of teams  
- $G$ : set of groups  
- $P$ : set of pots, where P = {1,2,3,4}
- $C$ : set of confederations  

---

### Parameters

- $p_t$ : FIFA ranking points of team $t$  
- $Pot(p)$ : set of teams belonging to pot $p$  
- $Confederation(c)$ : set of teams belonging to confederation $c$

---

### Decision Variables

$$
x_{tg} =
\begin{cases}
1 & \text{if team } t \text{ is assigned to group } g \\
0 & \text{otherwise}
\end{cases}
$$

$$
w = \text{minimum group strength}
$$

The binary variable $x_{tg}$ indicates whether team $t$ is assigned to group $g$. The variable $w$ represents the minimum total FIFA ranking points among all groups.

---

### Objective Function

The objective is to maximize the minimum group strength:

$$
\max w
$$

Maximizing $w$ ensures that the weakest group (in terms of FIFA ranking points) is as strong as possible, which leads to balanced groups.

---

### Constraints

Each team is assigned to exactly one group:

$$
\sum_{g \in G} x_{tg} = 1 \quad \forall t \in T
$$

Each group contains exactly four teams:

$$
\sum_{t \in T} x_{tg} = 4 \quad \forall g \in G
$$

Minimum group strength constraint:

$$
w \le \sum_{t \in T} p_t x_{tg} \quad \forall g \in G
$$

This constraint ensures that $w$ is less than or equal to the total FIFA points of every group and therefore represents the minimum group strength.

---

### Pot Constraint

Each group can contain **at most one team from each pot**:

$$
\sum_{t \in Pot(p)} x_{tg} \le 1 \quad \forall g \in G, \; p \in P
$$

This formulation ensures that no two teams from the same pot are assigned to the same group.

---

### Confederation Constraint

Teams from the same confederation must satisfy FIFA draw restrictions:

$$
\sum_{t \in Confederation(c)} x_{tg} \le
\begin{cases}
2 & \text{if } c = UEFA \\
1 & \text{otherwise}
\end{cases}
\quad \forall g \in G, \; c \in C
$$

This means that a group may contain **at most two UEFA teams**, while teams from all other confederations can appear **at most once in each group**.

---

### Symmetry Breaking Constraints

To reduce symmetric solutions and improve computational efficiency, two anchor constraints are introduced:

$$
x_{t_1,1} = 1
$$

$$
x_{t_2,2} = 1
$$

where $t_1$ and $t_2$ represent the first two teams in the ranking list. These constraints fix the top-ranked team to Group 1 and the second-ranked team to Group 2. They do not affect optimality but help reduce equivalent symmetric solutions.

---

### Model Implementation

The optimization model is implemented in **Python using the Pyomo optimization framework** and solved using the **GLPK solver**. The solver searches for the assignment of teams to groups that maximizes the minimum group strength while satisfying all tournament constraints.

In [1]:
# Install system package for GLPK
!apt-get update -qq
!apt-get install -y -qq glpk-utils libglpk-dev

# Install Python packages
!pip install -q pyomo openpyxl ortools

# Imports
import pandas as pd
import matplotlib.pyplot as plt

from pyomo.environ import *
from ortools.sat.python import cp_model

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
file_path = "AAMA groupwork.xlsx"
df = pd.read_excel(file_path, sheet_name="Confederation")

# Keep only necessary columns
df = df.loc[:, ["Team", "Confederation", "FIFA Points"]]

# Sort by FIFA points (descending)
df = df.sort_values(by="FIFA Points", ascending=False).reset_index(drop=True)

# Create pots based on ranking position (4 pots of 12 teams)
df["Pot"] = df.index // 12 + 1

teams = df["Team"].tolist()
points = df.set_index("Team")["FIFA Points"].to_dict()
confed = df.set_index("Team")["Confederation"].to_dict()
pots = df.set_index("Team")["Pot"].to_dict()

# Build the optimization model
num_groups = 12
groups = range(1, num_groups + 1)

model = ConcreteModel()

model.T = Set(initialize=teams)
model.G = Set(initialize=groups)

# Decision variable
model.x = Var(model.T, model.G, domain=Binary)

# w represents the minimum total FIFA points across all groups
model.w = Var(domain=NonNegativeReals)

# Objective: maximise the minimum group strength
model.obj = Objective(expr=model.w, sense=maximize)

# Constraints

# Ensure w is less than or equal to every group strength
def min_group_strength_rule(m, g):
    return m.w <= sum(points[t] * m.x[t, g] for t in m.T)

model.min_strength = Constraint(model.G, rule=min_group_strength_rule)

# Each group has exactly 4 teams
def group_size_rule(m, g):
    return sum(m.x[t, g] for t in m.T) == 4

model.group_size = Constraint(model.G, rule=group_size_rule)

# Each team assigned to exactly one group
def team_assignment_rule(m, t):
    return sum(m.x[t, g] for g in m.G) == 1

model.team_assignment = Constraint(model.T, rule=team_assignment_rule)

# Pot constraint: max 1 team per pot per group
def pot_constraint_rule(m, g, p):
    teams_in_pot = [t for t in m.T if pots[t] == p]
    return sum(m.x[t, g] for t in teams_in_pot) <= 1

model.Pots = Set(initialize=[1,2,3,4])
model.pot_constraint = Constraint(model.G, model.Pots, rule=pot_constraint_rule)

# Confederation constraint
# UEFA allows up to two teams per group
# Other confederations allow at most one team
confeds = sorted(set(confed.values()))
model.C = Set(initialize=confeds)

def confed_constraint_rule(m, g, c):
    teams_in_confed = [t for t in teams if confed[t] == c]
    if c == "UEFA":
        return sum(m.x[t, g] for t in teams_in_confed) <= 2
    else:
        return sum(m.x[t, g] for t in teams_in_confed) <= 1

model.confed_constraint = Constraint(model.G, model.C, rule=confed_constraint_rule)

# Solve using glpk
solver = SolverFactory("glpk")
solver.options["tmlim"] = 300

# Anchor constraints to break symmetry
# Fix top-ranked team to Group 1
# Fix second-ranked team to Group 2
top_team = teams[0]
model.anchor = Constraint(expr=model.x[top_team, 1] == 1)
second_team = teams[1]
model.anchor2 = Constraint(expr=model.x[second_team, 2] == 1)

results = solver.solve(model, tee=True)
print("status =", results.solver.status)
print("termination =", results.solver.termination_condition)

# Print results
print("\nBalanced Groups:\n")

for g in groups:
    group_teams = [t for t in teams if model.x[t, g].value == 1]
    total_points = sum(points[t] for t in group_teams)

    group_teams = sorted(group_teams, key=lambda t: pots[t])

    print(f"Group {g}:")
    for t in group_teams:
        print(f"   Pot {pots[t]} - {t} ({points[t]})")
    print(f"   Total Points: {total_points}\n")

print("Minimum Group Strength (w):", model.w.value)


GLPSOL: GLPK LP/MIP Solver, v4.65
Parameter(s) specified in the command line:
 --tmlim 300 --write C:\Users\kaust\AppData\Local\Temp\tmp_onaqy0o.glpk.raw
 --wglp C:\Users\kaust\AppData\Local\Temp\tmp6i78cqxg.glpk.glp --cpxlp C:\Users\kaust\AppData\Local\Temp\tmprupvvk6k.pyomo.lp
Reading problem data from 'C:\Users\kaust\AppData\Local\Temp\tmprupvvk6k.pyomo.lp'...
C:\Users\kaust\AppData\Local\Temp\tmprupvvk6k.pyomo.lp:4100: warning: lower bound of variable 'x4' redefined
C:\Users\kaust\AppData\Local\Temp\tmprupvvk6k.pyomo.lp:4100: warning: upper bound of variable 'x4' redefined
206 rows, 577 columns, 2894 non-zeros
576 integer variables, all of which are binary
4676 lines were read
Writing problem data to 'C:\Users\kaust\AppData\Local\Temp\tmp6i78cqxg.glpk.glp'...
3888 lines were written
GLPK Integer Optimizer, v4.65
206 rows, 577 columns, 2894 non-zeros
576 integer variables, all of which are binary
Preprocessing...
187 rows, 529 columns, 2640 non-zeros
528 integer variables, all of wh

## Model 2 – Letter Assignment



### Match Popularity

Following **Ghoniem et al. (2017)**, the popularity of a match between two teams is determined by the spectator indices of the two teams.

Let

- $f_i$ = spectator index of team $i$
- $f_j$ = spectator index of team $j$

The popularity of a match between team $i$ and team $j$ is defined as

$$
\pi_{ij}
=
f_i + f_j + \frac{f_i + f_j}{2} + \text{officials term}
$$

where the officials term is

$$
\text{officials} =
\begin{cases}
1, & \text{if } \max(f_i,f_j)=1 \\
\frac{f_i + f_j}{2}, & \text{otherwise}
\end{cases}
$$

Thus the complete match popularity function becomes

$$
\pi_{ij}
=
f_i + f_j
+
\frac{f_i + f_j}{2}
+
\begin{cases}
1 & \text{if } \max(f_i,f_j)=1 \\
\frac{f_i + f_j}{2} & \text{otherwise}
\end{cases}
$$

This formulation captures the expected spectator demand generated by both teams, neutral spectators, and match officials.

The objective of **Model 2** is to assign balanced groups to tournament letters (A–L) such that match popularity is distributed as evenly as possible across the tournament schedule.

---

### Sets

- $G$: set of balanced groups obtained from Model 1  
- $L$: set of tournament group letters $(A, B, \dots, L)$  
- $R$: set of row-sets in the tournament schedule  

A **row-set** represents a group of matches played simultaneously within the same time window.

---

### Parameters

- $c_{rgl}$: popularity contribution if group $g$ is assigned to letter $l$ in row-set $r$

For each row-set $r$, the tournament schedule specifies which **rank pairings** are played (for example rank 1 vs rank 2, rank 3 vs rank 4).

If group $g$ is assigned letter $l$, the matches of that group generate popularity values according to the match popularity formula.

Therefore,

$$
c_{rgl}
=
\sum_{(a,b)\in P(r,l)} \pi_{g,(a,b)}
$$

where $P(r,l)$ represents the set of rank pairings scheduled in row-set $r$ for letter $l$.

In the implementation, these coefficients are **pre-computed prior to solving the optimisation model**.

---

### Decision Variables

$$
z_{gl} =
\begin{cases}
1, & \text{if group } g \text{ is assigned letter } l \\
0, & \text{otherwise}
\end{cases}
$$

Row-set popularity is defined as

$$
y_r =
\sum_{g\in G}\sum_{l\in L} c_{rgl} z_{gl}
$$

which represents the total popularity of all matches occurring in row-set $r$.

---

### Objective Function

The objective is to maximise

$$
\max (w_{\min} + w_{\max})
$$

where

- $w_{\min}$ represents the **minimum row-set popularity**
- $w_{\max}$ represents the **maximum row-set popularity**

Maximising this objective simultaneously increases the popularity of the least attractive row-set while allowing highly popular matches to occur in the most attractive time slots.

---

### Constraints

Each balanced group must receive exactly one tournament letter:

$$
\sum_{l \in L} z_{gl} = 1
\qquad \forall g \in G
$$

Each tournament letter is assigned to exactly one group:

$$
\sum_{g \in G} z_{gl} = 1
\qquad \forall l \in L
$$

---

### Minimum Row-Set Popularity

$$
w_{\min} \le y_r
\qquad \forall r \in R
$$

This constraint ensures that $w_{\min}$ represents the smallest popularity among all row-sets.

---

### Maximum Row-Set Popularity

$$
w_{\max} \ge y_r
\qquad \forall r \in R
$$

This ensures that $w_{\max}$ is at least as large as every row-set popularity.

---

### Maximum Linearisation

To ensure that $w_{\max}$ correctly identifies the largest row-set popularity, the following linearisation is introduced:

$$
\sum_{r \in R} v_r = 1
$$

$$
w_{\max} \le y_r + M(1-v_r)
\qquad \forall r \in R
$$

where

- $v_r$ is a binary variable identifying the row-set that attains the maximum popularity  
- $M$ is a sufficiently large constant.

In [3]:
# Scale popularity values for CP-SAT
SCALE = 1_000_000
TIME_LIMIT = 60.0


def match_popularity(fi: float, fj: float, tol: float = 1e-9) -> float:
    # Match popularity formula from Ghoniem et al. (2017)
    fi = max(0.0, min(1.0, float(fi)))
    fj = max(0.0, min(1.0, float(fj)))
    other = 0.5 * (fi + fj)
    officials = 1.0 if max(fi, fj) >= 1.0 - tol else 0.5 * (fi + fj)
    return fi + fj + other + officials


def main():
    # Load input tables
    file_path = "AAMA groupwork.xlsx"
    groups = pd.read_excel(file_path, sheet_name="groups_long")
    grid = pd.read_excel(file_path, sheet_name="match_list_long")

    # Clean up data types
    groups["group_num"] = groups["group_num"].astype(int)
    groups["rank_in_group"] = groups["rank_in_group"].astype(int)
    grid["row_set_id"] = grid["row_set_id"].astype(int)
    grid["group_letter"] = grid["group_letter"].astype(str)
    grid["rank_a"] = grid["rank_a"].astype(int)
    grid["rank_b"] = grid["rank_b"].astype(int)

    # Precompute match popularity by group and rank pairing
    si = {
        (int(r.group_num), int(r.rank_in_group)): float(r.spectator_index)
        for r in groups.itertuples()
    }

    pi = {}
    for g in sorted(groups["group_num"].unique()):
        for a in [1, 2, 3]:
            for b in range(a + 1, 5):
                pi[(g, a, b)] = match_popularity(si[(g, a)], si[(g, b)])

    # Define sets
    G = sorted(groups["group_num"].unique())      # balanced groups
    L = sorted(grid["group_letter"].unique())     # letters A..L
    R = sorted(grid["row_set_id"].unique())       # row-sets

    # Set Big-M
    matches_per_r = grid.groupby("row_set_id").size().to_dict()
    n_max = int(max(matches_per_r.values()))
    M_big = int(round(4.0 * n_max * SCALE))

    # Precompute row-set popularity coefficients
    coef = {}
    for r in R:
        df_r = grid[grid["row_set_id"] == r]

        for letter in L:
            df_rl = df_r[df_r["group_letter"] == letter]
            pairs = []

            for rr in df_rl.itertuples():
                a, b = int(rr.rank_a), int(rr.rank_b)
                if a > b:
                    a, b = b, a
                pairs.append((a, b))

            for g in G:
                val = sum(pi[(g, a, b)] for (a, b) in pairs)
                coef[(r, g, letter)] = int(round(val * SCALE))

    # Build CP-SAT model
    model = cp_model.CpModel()

    # z[g, letter] = 1 if balanced group g is assigned to letter
    z = {(g, letter): model.NewBoolVar(f"z_{g}_{letter}") for g in G for letter in L}

    # Each group gets exactly one letter
    for g in G:
        model.Add(sum(z[(g, letter)] for letter in L) == 1)

    # Each letter is used exactly once
    for letter in L:
        model.Add(sum(z[(g, letter)] for g in G) == 1)

    # Row-set popularity y[r]
    y = {r: model.NewIntVar(0, M_big, f"y_{r}") for r in R}

    for r in R:
        expr = []
        for letter in L:
            for g in G:
                c = coef[(r, g, letter)]
                if c != 0:
                    expr.append(z[(g, letter)] * c)
        model.Add(y[r] == (sum(expr) if expr else 0))

    # Min / max row-set popularity
    w_min = model.NewIntVar(0, M_big, "w_min")
    w_max = model.NewIntVar(0, M_big, "w_max")

    for r in R:
        model.Add(w_min <= y[r])
        model.Add(w_max >= y[r])

    # Paper-style max linearisation
    v = {r: model.NewBoolVar(f"v_{r}") for r in R}
    model.Add(sum(v[r] for r in R) == 1)

    for r in R:
        model.Add(w_max <= y[r] + M_big * (1 - v[r]))

    # Objective
    model.Maximize(w_min + w_max)

    # Solve model
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = TIME_LIMIT
    solver.parameters.num_search_workers = 8

    status = solver.Solve(model)

    if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
        print("No feasible solution found.")
        return None, None, None

    # Extract results
    group_to_letter = {
        g: next(letter for letter in L if solver.Value(z[(g, letter)]) == 1)
        for g in G
    }

    y_val = {r: solver.Value(y[r]) / SCALE for r in R}

    assign_df = pd.DataFrame(
        [{"group_num": g, "assigned_letter": group_to_letter[g]} for g in G]
    ).sort_values("assigned_letter").reset_index(drop=True)

    row_df = pd.DataFrame(
        [{
            "row_set_id": r,
            "num_matches": matches_per_r[r],
            "rowset_popularity_y": y_val[r]
        } for r in R]
    ).sort_values("row_set_id").reset_index(drop=True)


    rowset_popularity = dict(zip(row_df["row_set_id"], row_df["rowset_popularity_y"]))

    # Print results
    print(f"Solver status: {solver.StatusName(status)}")
    print(f"w_min: {solver.Value(w_min) / SCALE:.6f}")
    print(f"w_max: {solver.Value(w_max) / SCALE:.6f}")
    print(f"objective (w_min + w_max): {(solver.Value(w_min) + solver.Value(w_max)) / SCALE:.6f}")

    print("\nLetter assignment")
    print(assign_df.to_string(index=False))

    print("\nRow-set popularity")
    print(row_df.to_string(index=False))
    return row_df, assign_df, rowset_popularity


row_df, assign_df, rowset_popularity = main()

Solver status: OPTIMAL
w_min: 3.848146
w_max: 12.873930
objective (w_min + w_max): 16.722076

Letter assignment
 group_num assigned_letter
         3               A
        10               B
        11               C
         4               D
         9               E
         2               F
        12               G
         8               H
         6               I
         7               J
         1               K
         5               L

Row-set popularity
 row_set_id  num_matches  rowset_popularity_y
          1            5             3.873107
          2            4             4.188070
          3            5             6.245271
          4            5             3.908732
          5            4             4.550208
          6            3             3.848146
          7            3             4.754768
          8            5             4.571926
          9            5            12.873930
         10            4             5.046221
         11

# Model 3 – Stadium Assignment

The final optimisation model assigns each row-set to a stadium in order to maximise total spectator exposure throughout the tournament.

In this implementation, the popularity of each row-set is taken from the output of Model 2, and stadium capacity is taken from the stadium dataset. High-popularity row-sets are therefore allocated to larger-capacity stadiums whenever this improves the total objective value.



The final optimisation assigns matches to stadiums in order to maximise spectator exposure.

---

### Sets
- $R$: set of row-sets
- $S$: set of stadiums

---

### Parameters
- $y_r$: popularity of row-set $r$
- $K_s$: capacity of stadium $s$

---

### Decision Variable


$$
x_{rs}=
\begin{cases}
1, & \text{if row-set } r \text{ is assigned to stadium } s,\\
0, & \text{otherwise.}
\end{cases}
$$


---

### Objective Function


$$
\max \sum_{r \in R}\sum_{s \in S} y_r K_s x_{rs}
$$

---

### Constraints

Each row-set is assigned to exactly one stadium

$$
\sum_{s \in S} x_{rs}=1
\qquad \forall r \in R
$$


Each stadium is used exactly once

$$
\sum_{r \in R} x_{rs}=1
\qquad \forall s \in S
$$


In [4]:
# Read input data
file_path_groupwork = "AAMA groupwork.xlsx"
stad_df = pd.read_excel(file_path_groupwork, sheet_name="Stadium")

# Define sets and parameters
R = row_df["row_set_id"].tolist()
S = stad_df["Stadium Name"].tolist()

y = dict(zip(R, pd.to_numeric(row_df["rowset_popularity_y"])))
cap = dict(zip(S, pd.to_numeric(stad_df["Capacity"])))

# Build model
model = ConcreteModel()
model.R = Set(initialize=R)
model.S = Set(initialize=S)
model.x = Var(model.R, model.S, domain=Binary)

# Objective function
model.obj = Objective(
    expr=sum(y[r] * cap[s] * model.x[r, s] for r in model.R for s in model.S),
    sense=maximize
)

# Constraints
def row_assign_rule(model, r):
    return sum(model.x[r, s] for s in model.S) == 1

def stadium_use_rule(model, s):
    return sum(model.x[r, s] for r in model.R) == 1

model.row_assign = Constraint(model.R, rule=row_assign_rule)
model.stadium_use = Constraint(model.S, rule=stadium_use_rule)

# Solve model
solver = SolverFactory("glpk")
results = solver.solve(model, tee=False)

print("status =", results.solver.status)
print("termination =", results.solver.termination_condition)
print("objective =", value(model.obj))

solution = []
for r in model.R:
    for s in model.S:
        if value(model.x[r, s]) > 0.5:
            solution.append([r, s, y[r], cap[s]])

sol_df = pd.DataFrame(
    solution,
    columns=["row_set_id", "stadium", "rowset_popularity_y", "capacity"]
).sort_values("row_set_id")

print("\nstadium assignment:")
print(sol_df.to_string(index=False))

status = ok
termination = optimal
objective = 5338812.511725

stadium assignment:
 row_set_id                         stadium  rowset_popularity_y  capacity
          1                   Estadio Akron             3.873107     44330
          2                     Lumen Field             4.188070     65123
          3                  Estadio Azteca             6.245271     72766
          4                        BC Place             3.908732     48821
          5           Mercedes-Benz Stadium             4.550208     67382
          6                       BMO Field             3.848146     44315
          7                     NRG Stadium             4.754768     68311
          8 GEHA Field at Arrowhead Stadium             4.571926     67513
          9                 MetLife Stadium            12.873930     78576
         10                  Levi's Stadium             5.046221     69391
         11         Lincoln Financial Field             4.405963     65827
         12       

# Overall Optimisation Results
The three optimisation models collectively produce a tournament schedule that:

- Maintains competitive balance between groups
- Distributes match popularity across the tournament schedule
- Maximises spectator exposure through stadium allocation

Together, these models form a sequential optimisation framework for tournament planning, linking competitive balance, match popularity, and stadium capacity.

